# MMM Scenario Planner — Colab Runner

Loads a trained model from an existing GCS run, generates budget optimization dataframes, and uploads CSVs to `clients/{client}/optimizations/{opt_id}/`.

**Before running:** push any local changes to GitHub — this notebook always pulls fresh from main.

**To run a different client or run:** edit `CLIENT` and `RUN_ID` in Cell 5, then Runtime → Run all.

In [ ]:
# Cell 0: Runtime check — requires T4 GPU + High RAM. Stops here if not met.
import subprocess, psutil, sys

errors = []

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5
    )
    if result.returncode == 0 and result.stdout.strip():
        print(f'GPU: {result.stdout.strip()}')
    else:
        errors.append('No GPU detected — switch to T4 GPU (Runtime > Change runtime type).')
except Exception:
    errors.append('Could not detect GPU — switch to T4 GPU (Runtime > Change runtime type).')

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
if ram_gb >= 40:
    print(f'RAM: {ram_gb:.0f} GB (High RAM confirmed)')
else:
    errors.append(f'Standard RAM detected ({ram_gb:.0f} GB) — enable High RAM (Runtime > Change runtime type).')

if errors:
    print('\nSTOP — fix the following before running:')
    for e in errors:
        print(f'  ✗ {e}')
    raise SystemExit(1)

print('Runtime OK — proceed.')

In [ ]:
# Cell 1: Clone repo (safe to re-run — pulls latest if already cloned)
import os
REPO = "/content/Meridian-MMM-System"
if os.path.isdir(REPO):
    !git -C {REPO} pull origin main
else:
    !git clone https://github.com/Russ-Moonride/Meridian-MMM-System.git {REPO}
%cd {REPO}

In [ ]:
# Cell 2: Install scenarioplanner extras
# The batch_size proto mismatch is handled by a runtime patch in the script.
!pip install google-meridian[scenarioplanner] -q

In [ ]:
# Cell 3: Mount Drive and download service account credentials
import gdown
from google.colab import drive
drive.mount('/content/drive')

file_id = '1Nnvt5h7QW6hC1R-sCiqXSgKvLclR767z'
destination_path = '/content/Meridian-MMM-System/service_account.json'

gdown.download(id=file_id, output=destination_path, quiet=False)

In [ ]:
# Cell 4: Set environment variables
import os
SA_PATH = '/content/Meridian-MMM-System/service_account.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = SA_PATH
# CUDA_VISIBLE_DEVICES intentionally unset — GPU is required for this notebook

In [ ]:
# Cell 5: Configure run — edit CLIENT and RUN_ID as needed
CLIENT = "northspore"
RUN_ID = "prod_2026-06-17_1913"   # must match an existing run folder in GCS

# Optional overrides (leave as-is for defaults)
OPTIMIZATION_NAME = "default"     # labels this scenario in the output folder name
FILTER_START      = "2024-01-01"  # drop data before this date to keep file sizes lean
MIN_SPEND_SHIFT   = 1.0           # 1.0 = no decrease below historical
MAX_SPEND_SHIFT   = 1.0           # 1.0 = no increase above historical; set >1 to allow reallocation
MONTHLY           = True
QUARTERLY         = False
YEARLY            = False

In [ ]:
# Cell 6: Run the scenario planner
monthly_flag   = "--monthly"    if MONTHLY   else "--no-monthly"
quarterly_flag = "--quarterly"  if QUARTERLY else ""
yearly_flag    = "--yearly"     if YEARLY    else ""

!GOOGLE_APPLICATION_CREDENTIALS={SA_PATH} python scripts/run_scenario_planner.py \
    --client            {CLIENT} \
    --run-id            {RUN_ID} \
    --optimization-name {OPTIMIZATION_NAME} \
    --filter-start      {FILTER_START} \
    --min-spend-shift   {MIN_SPEND_SHIFT} \
    --max-spend-shift   {MAX_SPEND_SHIFT} \
    {monthly_flag} {quarterly_flag} {yearly_flag}

In [ ]:
# Cell 7: Confirm GCS output
from google.cloud import storage
import yaml, os

with open(f'configs/{CLIENT}.yaml') as f:
    cfg = yaml.safe_load(f)

gcs_runs_base = cfg['gcs_output_path'].rstrip('/')
gcs_opt_base  = gcs_runs_base.replace('/runs', '/optimizations')

bucket_name = gcs_opt_base.split('gs://')[1].split('/')[0]
prefix = '/'.join(gcs_opt_base.split('gs://')[1].split('/')[1:])

client = storage.Client()
blobs = list(client.list_blobs(bucket_name, prefix=prefix))
print(f"Files in {gcs_opt_base}/")
for b in blobs:
    print(f"  gs://{bucket_name}/{b.name}")